# Step 0 - Customer Facing Assistant Demo

- This notebook is the presenter entry point for a local assistant built on synthetic customer, order, instrument, service, and public troubleshooting records.
- Step 1 validates six CSV files and loads a disposable SQLite copy. The source datasets are never edited.
- Step 2 turns the selected Alice/Bob demo login into a short-lived signed user identity; private reads use that identity to check record ownership.
- Step 3 starts the MCP server with `get_order_status` for one owned order, `get_service_history` for an owned instrument, and `search_troubleshooting` for approved public knowledge.
- Step 4 chooses OpenAI or Anthropic through one gateway interface. The default notebook run uses an offline fake, so no provider key or network call is needed.
- Step 5 sends only authorized evidence to the gateway and shows the answer, citations, trace ID, and model route. Step 14 lets the live model select reviewed MCP tools from their discovered schemas.
- Step 6 stores tool decisions and model responses in an audit trace, derives the highest risk tier touched, and rejects an intentionally false candidate in the evaluation gate.
- Step 7 provides the timed walkthrough and links to the architecture and handoff notes.
- Step 8 adds a Streamlit front end over the same orchestrator, with a selected demo customer and live model provider.
- Step 9 adds a fourth MCP tool, `list_customer_orders`, so an ID-free order request returns every order owned by the selected signed customer.
- Step 10 adds continuous customer-separated chat and a 72-hour local audit trace history in Streamlit; the A2A trace label describes lineage, not a completed A2A handoff.
- Step 11 joins authorized order/service rows to their customer record, checks requested order dates against `created_on`, lets `get_service_history` list all owned service events when no instrument ID is given, and lets `search_troubleshooting` list approved public articles when no search terms are given.
- Step 14 discovers the reviewed MCP catalog, validates model-proposed arguments, and executes at most four calls in three planning rounds. The explicit offline fake uses deterministic rehearsal rules; live failures never activate them.

Select this project's `.venv` kernel, restart it, and run all cells. The default settings make the walkthrough repeatable offline.


In [ ]:
# Step 0: make the local package importable when a fresh kernel starts in
# either the project folder or the notebooks folder.
from pathlib import Path
import sys

# Add the local package path so a fresh kernel works without installing this package.
project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src' / 'customer_assistant').is_dir())
sys.path.insert(0, str(project_root / 'src'))
# Import before any data or model work so a wrong kernel/path fails with a
# clear setup error instead of looking like an MCP or provider failure.
import customer_assistant
print(f'Package import OK: {customer_assistant.__version__}')

## Step 5 - Presenter settings

Edit only the settings in the next cell before **Run All**. `demo_user` is a simulated login, not a model-supplied identity. `provider = None` follows `.env` or the OpenAI default; an explicit `"openai"` or `"anthropic"` changes the gateway route. `demo_mode = "offline"` always uses evidence-based synthetic replies and makes no provider call, even if a key exists. `demo_mode = "live"` uses the selected provider for bounded tool selection and the final cited answer to the configured question; it skips the curated examples.


In [ ]:
# Step 5: keep presenter choices in one cell so rerunning a scenario does not
# require editing the orchestrator. These values never become identity claims.
demo_user = 'alice'
provider = None  # None follows DEMO_MODEL_PROVIDER; explicit values test the gateway route.
question = 'Where is DEMO-ORD-1007?'
instrument_model = '1260 Infinity II Quaternary Pump'
demo_mode = 'offline'  # Live mode makes an intentional external model call.

from customer_assistant.config import load_gateway_config
gateway_config = load_gateway_config(provider=provider)
if demo_mode not in {'offline', 'live'}:
    raise ValueError("demo_mode must be 'offline' or 'live'")
# Show the selected route without printing the GatewayConfig object, which
# can hold a provider API key loaded from the process or .env file.
print(f'Demo settings: user={demo_user}, provider={gateway_config.provider}, model={gateway_config.model}, mode={demo_mode}')
print(f'Question: {question}')

In [ ]:
# Step 5: prepare the generated SQLite copy and one signing secret. The
# secret lets the separate MCP process verify that private tool calls carry
# identity minted by this application, not an ID supplied by a prompt.
import os
import secrets
from dotenv import dotenv_values
from customer_assistant.database import DEFAULT_DB_PATH, load_database

# A fresh checkout excludes data/local, so load a new validated copy only
# when the generated database is absent; do not edit the source CSVs.
if not DEFAULT_DB_PATH.is_file():
    load_database()
# Honor an explicit process secret; otherwise use .env or generate an
# ephemeral demo-only key so offline Run All works without setup.
# The key stays in process memory so the MCP subprocess can verify signatures.
if not os.environ.get('DEMO_SIGNING_SECRET'):
    env_secret = dotenv_values(project_root / '.env').get('DEMO_SIGNING_SECRET')
    os.environ['DEMO_SIGNING_SECRET'] = env_secret or secrets.token_hex(32)
print('Synthetic data and demo identity are ready.')

In [ ]:
# Step 5/14: the live model proposes reads from the discovered MCP catalog.
# The orchestrator validates arguments, supplies signed identity, and
# checks fresh evidence before re-planning or generating a cited answer.
from customer_assistant.gateway import LiteLLMGateway
from customer_assistant.orchestrator import answer_question, make_offline_demo_gateway

def show_result(label, result):
    # Keep the presenter output reviewable without dumping signed identity,
    # raw private tool rows, provider keys, or transport details.
    item = result.as_dict()
    print(f'\n{label}')
    print(f"Outcome: {item['outcome']} | Skill: {item['skill']}")
    print(f"Answer: {item['answer_text']}")
    print(f"Sources: {', '.join(item['source_ids']) or '(none)'}")
    print(f"Trace: {item['trace_id']}")
    print(f"Provider/model: {item['provider']} / {item['model']}")
    print(f"Token usage: {item['token_usage']}")

# Both gateways consume the same authorized evidence projection. Use
# the offline fake by default; construct a live adapter only when requested.
if demo_mode == 'live' and not gateway_config.api_key:
    raise ValueError(f'Set {gateway_config.provider.upper()}_API_KEY for live mode')
gateway = make_offline_demo_gateway(gateway_config) if demo_mode == 'offline' else LiteLLMGateway(gateway_config)
configured_result = await answer_question(
    demo_user, question, gateway, db_path=DEFAULT_DB_PATH,
    instrument_model=instrument_model,
)
show_result('Configured question', configured_result)

## Step 5 - Offline walkthrough

The next cell shows four distinct authorization paths. Alice's order and service questions may use her private rows; Bob's request for Alice's order must be denied before any row or model response is exposed; the pressure warning reads only approved public material. Each example still calls the separate MCP server over stdio. Only the model answer is simulated.


In [ ]:
# Step 5: exercise the four policy paths with the same orchestrator.
# This makes the contrast between private authorization and public knowledge
# visible without changing MCP tools or model messages by hand.
if demo_mode == 'offline':
    examples = (
        ('Alice order', 'alice', 'Where is DEMO-ORD-1007?'),
        ('Bob denied', 'bob', 'Where is DEMO-ORD-1007?'),
        ('Alice service', 'alice', 'What is the service history for DEMO-INS-1001?'),
        ('Public pressure warning', 'alice', 'What does Pressure Below Lower Limit mean for my 1260 Infinity II Quaternary Pump?'),
    )
    bob_result = None  # Keep the denial trace for the Step 6 lineage view.
    for label, user, example_question in examples:
        # Reuse the configured answer when it is the same case to avoid an
        # unnecessary duplicate MCP call. Other cases each receive a new
        # signed trace and separate MCP tool call.
        result = (configured_result if user == demo_user and example_question == question
                  else await answer_question(user, example_question, gateway,
                                             db_path=DEFAULT_DB_PATH, instrument_model=instrument_model))
        show_result(label, result)
        if label == 'Bob denied':
            bob_result = result
else:
    print('Offline examples skipped in live mode.')

In [ ]:
# Step 5: compare two provider configurations on the same order question.
# A changing answer would make the seam hard to inspect, so both use the
# deterministic evidence-aware fake while MCP authorization still runs.
# Both runs use the explicit offline selector; neither calls a provider API.
if demo_mode == 'offline':
    comparison_question = 'Where is DEMO-ORD-1007?'
    for comparison_provider in ('openai', 'anthropic'):
        comparison_config = load_gateway_config(provider=comparison_provider)
        comparison_gateway = make_offline_demo_gateway(comparison_config)
        comparison_result = await answer_question(
            'alice', comparison_question, comparison_gateway,
            db_path=DEFAULT_DB_PATH, instrument_model=instrument_model,
        )
        show_result(f'Offline provider comparison: {comparison_provider}', comparison_result)
else:
    print('Offline provider comparison skipped in live mode.')

## Step 6 - Evaluation and promotion gate

The offline harness reruns the normal assistant cases and checks cross-customer denial, factual grounding, complete action traces, and the test set required for the inherited risk tier. EVAL-005 separately tests a *candidate-only* claim that Alice's in-transit order was delivered. The candidate is given a valid source marker so the fact check must catch the contradiction itself. A passing harness run confirms that the expected failure was found; the promotion decision stays **blocked**.


In [ ]:
# Step 6: run the fixed synthetic gate set only in offline mode so a live
# provider answer is never confused with the deterministic evaluation cases.
# The wrong EVAL-005 text is created and checked only in evaluation; it
# never becomes an AssistantResult shown to a customer.
import json
from customer_assistant.evaluation import run_evaluation

if demo_mode == 'offline':
    evaluation_report = await run_evaluation(db_path=DEFAULT_DB_PATH)
    evaluation_data = evaluation_report.as_dict()
    print('Evaluation gates:')
    print(json.dumps(evaluation_data['gates'], indent=2))
    print(f"Harness status: {evaluation_data['harness_status']}")
    print(f"Promotion: {evaluation_data['promotion']}")
    case_rows = evaluation_data['cases']
    if isinstance(case_rows, dict):
        case_rows = case_rows.values()
    wrong_candidate = next((case for case in case_rows if case.get('case_id') == 'EVAL-005'), None)
    if wrong_candidate is None:
        raise AssertionError('EVAL-005 is missing from the evaluation report')
    print('EVAL-005 candidate-only result:')
    print(json.dumps(wrong_candidate, indent=2))
    if 'fail' not in str(wrong_candidate).casefold():
        raise AssertionError('EVAL-005 did not fail as intended')
    if 'blocked' not in str(evaluation_data['promotion']).casefold():
        raise AssertionError('Promotion did not remain blocked')
else:
    print('Offline evaluation skipped in live mode.')

## Step 6 - Trace inspection

The next cell reads both the configured request and Bob's denied request by trace ID. Each event identifies the accountable user, agent, action, outcome, source IDs, provider/model, prompt version, and computed tier. The database keeps the authorized evidence snapshot and normalized model response for reconstruction. This presenter view prints only the snapshot row count and whether a model answer was recorded, so Bob's empty denial is easy to see without displaying private rows or credentials.


In [ ]:
# Step 6: inspect one allowed and one denied trace. The audit database
# retains snapshots for reconstruction, but the presenter view exposes only
# counts and metadata, not the private evidence itself.
from customer_assistant.audit import read_trace

if demo_mode == 'offline':
    # The count shows whether an authorized snapshot was retained. Bob's
    # count must
    # remain zero, and his trace must not contain a model response.
    visible_fields = ('timestamp_utc', 'user_id', 'agent_id', 'action', 'outcome',
                      'evidence_ids', 'provider', 'model', 'prompt_version', 'risk_tier')
    for label, result in (('Configured', configured_result), ('Bob denied', bob_result)):
        trace_events = read_trace(result.trace_id, db_path=DEFAULT_DB_PATH)
        if not trace_events:
            raise AssertionError(f'No audit events found for {label} trace')
        safe_events = [
            {**{key: event.get(key) for key in visible_fields},
             'authorized_evidence_rows': len(event['authorized_evidence_snapshot']),
             'model_answer_recorded': event['response_text'] is not None}
            for event in trace_events
        ]
        print(f'{label} trace: {result.trace_id}')
        print(json.dumps(safe_events, indent=2))
else:
    print('Offline trace inspection skipped in live mode.')


## Step 7 - 10-12 minute panel rehearsal

- **0:00-1:00:** Introduce the synthetic scope and name the four MCP tools: one order, all owned orders, owned service history (one instrument or all), and approved public troubleshooting (matching articles or the catalog).
- **1:00-3:30:** Show Alice's owned order and service results. Point to the supported status, citations, trace IDs, and different private tiers.
- **3:30-4:45:** Show the public pressure article, manual locator, and escalation advice.
- **4:45-6:00:** Show Bob's denial for Alice's order and its empty evidence list.
- **6:00-7:30:** Switch the offline provider setting on the same question. The same MCP and entitlement path runs with a different gateway label. A true live swap requires verified keys and routes; run the configured question once per provider in `live` mode if available.
- **7:30-9:00:** Show EVAL-005's cited false claim failing factual grounding while the harness passes and promotion remains blocked.
- **9:00-10:30:** Compare the allowed trace with Bob's denied trace, including snapshot counts and the absence of a Bob model answer.
- **10:30-12:00:** Summarize the OIDC/JWT and CRM production stubs, risk ownership, and why the local audit trace is only an A2A handoff foundation for this single-orchestrator slice.

Use `docs/demo_rehearsal.md` for clean-start commands and `docs/architecture_and_operating_model.md` for the written architecture and operating-model handoff.
